Logs

In [1]:
%run Utils_Log

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 3, Finished, Available, Finished, True)

Utils_Log cargado. Llama a setup_log('nombre_del_stage') para empezar.


In [2]:
setup_log("silver_historico")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 4, Finished, Available, Finished, False)

[2026-06-09T10:08:50] ======================================================================
[2026-06-09T10:08:50] INICIO de la fase: silver_historico
[2026-06-09T10:08:50] Archivo de log:    Files/Control/logs/silver_historico/silver_historico_20260609_100850.log
[2026-06-09T10:08:50] ======================================================================


### Catálogos de validación (data contract v1.0)

Catálogos oficiales y rango temporal de referencia que usa el bloque de
aplicación del data contract (sección 7): provincias costeras, zonas de
pesca de bajura y el periodo 2015-2025 que cubren el modelo y el dashboard.

In [3]:
PROVINCIAS_VALIDAS = ["A Coruña", "Lugo", "Pontevedra"]

ZONAS_VALIDAS = [
    "Zona I - Vigo",
    "Zona II - Pontevedra",
    "Zona III - Arousa",
    "Zona IV - Muros",
    "Zona V - Fisterra",
    "Zona VI - Costa da Morte",
    "Zona VII - Coruña-Ferrol",
    "Zona VIII - Cedeira",
    "Zona IX - Mariña",
]

FECHA_MIN = "2015-01-01"
FECHA_MAX = "2025-12-31"

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 5, Finished, Available, Finished, False)

# Normalización de texto

1. **Normalización de `lonxa`**
   - Eliminación de espacios al inicio y final (`trim`).
   - Eliminación de puntos finales cuando existan (por ejemplo, `Ribeira.` → `Ribeira`).

2. **Limpieza de columnas de texto**
   - Aplicación de `trim` sobre `grupobiologico`, `fao`, `especie`, `provincia`, `zona` y `lonxa` para evitar discrepancias causadas por espacios sobrantes.

3. **Unificación de nombres de especie**
   - Se utiliza el código `FAO` como identificador único de especie.
   - Para cada código FAO se toma la primera denominación encontrada cronológicamente y se considera el nombre canónico.
   - Posteriormente todos los registros con el mismo FAO adoptan ese nombre, corrigiendo diferencias de escritura o nomenclatura (por ejemplo, `Anguila`, `Anguila`, `Anguías`).
   - Esta transformación no modifica la clasificación biológica ni elimina registros; únicamente estandariza la representación textual de una misma especie.

El objetivo de esta sección es asegurar que valores equivalentes tengan una única representación en el dataset, facilitando posteriores procesos de validación, agregación y análisis.


In [36]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col,upper , lit, length ,regexp_replace, when, round, trim, expr, broadcast, to_date, coalesce, isnull, count, month, row_number, first, current_timestamp
from pyspark.sql.types import FloatType, IntegerType
from pyspark.sql.functions import year


# 1. EXTRACCIÓN (Desde la ingesta)

df_raw = spark.read \
    .option("header", "false") \
    .option("delimiter", ";") \
    .csv("Files/Bronze/Ventas/anio=*/*.csv")

columnas_reales = [
    "data", "grupobiologico", "fao", "especie", "provincia", 
    "zona", "lonxa", "cantidad", "importe", "precio"
]

df_raw = df_raw.toDF(*columnas_reales)
log(f"Extracción completada: {df_raw.count()} filas leídas")


# 2. TRANSFORMACIÓN ESTRUCTURAL de tipos

log("Transformación: casting de tipos y conversión de comas decimales")
df_silver = df_raw \
    .withColumn("data", to_date(col("data"), "dd/MM/yyyy")) \
    .withColumn("grupobiologico", col("grupobiologico")) \
    .withColumn("fao", col("fao")) \
    .withColumn("especie", col("especie")) \
    .withColumn("provincia", col("provincia")) \
    .withColumn("zona", col("zona")) \
    .withColumn("cantidad", regexp_replace(col("cantidad"), ",", ".").cast(FloatType())) \
    .withColumn("importe", regexp_replace(col("importe"), ",", ".").cast(FloatType())) \
    .withColumn("precio", regexp_replace(col("precio"), ",", ".").cast(FloatType()))
    
df_silver = df_silver.withColumn("anio_archivo", year(col("data")))


# 3. LIMPIEZA DE RUIDO (Aplicación del Análisis Exploratorio)

df_silver = df_silver.na.drop(subset=["importe", "cantidad"])

# Corrección de anomalías tipográficas en memoria distribuida

df_silver = df_silver.withColumn(
    "lonxa", 
    regexp_replace(trim(col("lonxa")), r"\.$", "")
)
log("Trim aplicado a todas las columnas de texto")


#display(df_silver.limit(5))



StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 42, Finished, Available, Finished, False)

[2026-06-09T10:30:15] Extracción completada: 2131500 filas leídas
[2026-06-09T10:30:16] Transformación: casting de tipos y conversión de comas decimales
[2026-06-09T10:30:16] Trim aplicado a todas las columnas de texto


Correccion de los nombres de Fao

In [37]:
_w = Window.partitionBy("fao").orderBy(col("data").asc()) # para cada fao ordena los registros por fecha

df_especie_norm = (
    df_silver
    .withColumn("_rn", row_number().over(_w)) 
    .filter(col("_rn") == 1) # nos quedamos con la primera fila del grupo fao
    .select(col("fao").alias("_fao_lookup"), col("especie").alias("_especie_canonica"))
)

df_silver = (
    df_silver
    .join(df_especie_norm, col("fao") == col("_fao_lookup"), "left") # se añaden l (dnd fao sea igual a fao_lokup)
    .withColumn("especie", col("_especie_canonica")) # la col especie vale lo mismo q especie_canonica
    .drop("_fao_lookup", "_especie_canonica")
)
log("Corrección de nombres diferentes para un mismo FAO")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 43, Finished, Available, Finished, False)

[2026-06-09T10:30:19] Corrección de nombres diferentes para un mismo FAO


In [38]:
from pyspark.sql.functions import countDistinct

# fao con mas de un valor distinto de grupobiologico -- ayuda de
# conformidad para la regla de calidad 'grupobiologico_inconsistente'
# (seccion 7). Marca el CODIGO fao completo, no filas concretas: no se
# presupone cual valor es el correcto. No se aplica ningun cambio
# todavia sobre los datos de negocio.
_fao_inconsistentes = (
    df_silver
    .groupBy("fao")
    .agg(countDistinct("grupobiologico").alias("_n_valores_grupo"))
    .filter(col("_n_valores_grupo") > 1)
    .select(col("fao").alias("_fao_marcado"))
)

df_silver = (
    df_silver
    .join(_fao_inconsistentes, col("fao") == col("_fao_marcado"), "left")
    .withColumn("_grupobiologico_inconsistente", col("_fao_marcado").isNotNull())
    .drop("_fao_marcado")
)
log("Columna _grupobiologico_inconsistente añadida (marca el fao COMPLETO si presenta >1 valor de grupobiologico, sin presuponer cuál es correcto)")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 44, Finished, Available, Finished, False)

[2026-06-09T10:30:21] Columna _grupobiologico_inconsistente añadida (marca el fao COMPLETO si presenta >1 valor de grupobiologico, sin presuponer cuál es correcto)


In [39]:
total_filas = df_silver.count()
print(f"Total de registros: {total_filas}")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 45, Finished, Available, Finished, False)

Total de registros: 2131500


In [40]:
df_silver.printSchema()

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 46, Finished, Available, Finished, False)

root
 |-- data: date (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- fao: string (nullable = true)
 |-- especie: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- precio: float (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- _grupobiologico_inconsistente: boolean (nullable = false)



In [41]:
df_silver.select([
    count(when(isnull(c), c)).alias(c) for c in df_silver.columns
]).show()


StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 47, Finished, Available, Finished, False)

+----+--------------+---+-------+---------+----+-----+--------+-------+------+------------+-----------------------------+
|data|grupobiologico|fao|especie|provincia|zona|lonxa|cantidad|importe|precio|anio_archivo|_grupobiologico_inconsistente|
+----+--------------+---+-------+---------+----+-----+--------+-------+------+------------+-----------------------------+
|   0|             0|  0|      0|        0|   0|    0|       0|      0|     0|           0|                            0|
+----+--------------+---+-------+---------+----+-----+--------+-------+------+------------+-----------------------------+



## Redondeo a 2 decimales

In [42]:
from pyspark.sql.functions import round as spark_round

df_silver = df_silver \
    .withColumn("cantidad", spark_round("cantidad", 2)) \
    .withColumn("importe",  spark_round("importe",  2)) \
    .withColumn("precio",   spark_round("precio",   2))

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 48, Finished, Available, Finished, False)

## Añadir columna fecha de ingesta/transformación

Añade la marca temporal de procesamiento. Se incorpora antes del
etiquetado para que quede registrada también en los datasets de
quarantine y reject, no solo en silver.

In [43]:
df_silver = df_silver.withColumn("fecha_ingesta", current_timestamp())
log("Columna fecha_ingesta añadida")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 49, Finished, Available, Finished, False)

[2026-06-09T10:30:57] Columna fecha_ingesta añadida


## Schema y muestra

In [44]:
df_silver.printSchema()
log("Schema printado")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 50, Finished, Available, Finished, False)

root
 |-- data: date (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- fao: string (nullable = true)
 |-- especie: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- precio: float (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- _grupobiologico_inconsistente: boolean (nullable = false)
 |-- fecha_ingesta: timestamp (nullable = false)

[2026-06-09T10:30:57] Schema printado


Comprobaciones informativas sobre el dataset ya conformado: formato del
código `fao` (longitud y mayúsculas), coherencia entre `data` y
`anio_archivo`, y conteo de nulos por columna. No enrutan registros —
verifican que el casting y la normalización se aplicaron correctamente
antes de evaluar las reglas de negocio.

In [45]:
# 8.1.a — fao debe tener exactamente 3 caracteres y ser mayúsculas
df_fao_ko = df_silver.filter(
    (length(col("fao")) != 3) | (col("fao") != upper(col("fao")))
)
print(f"Registros con fao incorrecto: {df_fao_ko.count()}")
df_fao_ko.select("fao").distinct().show(50, truncate=False)

# 8.1.b — el año extraído de data debe coincidir con anio_archivo
df_anio_ko = df_silver.filter(year(col("data")) != col("anio_archivo"))
print(f"Registros con año inconsistente: {df_anio_ko.count()}")

# 8.1.c — conteo de nulos por columna
_nulls_df = df_silver.select([
    count(when(isnull(c), c)).alias(c) for c in df_silver.columns
])
_nulls_df.show()
log(f"Conteo de nulos por columna: {_nulls_df.collect()[0].asDict()}")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 51, Finished, Available, Finished, False)

Registros con fao incorrecto: 0
+---+
|fao|
+---+
+---+

Registros con año inconsistente: 0
+----+--------------+---+-------+---------+----+-----+--------+-------+------+------------+-----------------------------+-------------+
|data|grupobiologico|fao|especie|provincia|zona|lonxa|cantidad|importe|precio|anio_archivo|_grupobiologico_inconsistente|fecha_ingesta|
+----+--------------+---+-------+---------+----+-----+--------+-------+------+------------+-----------------------------+-------------+
|   0|             0|  0|      0|        0|   0|    0|       0|      0|     0|           0|                            0|            0|
+----+--------------+---+-------+---------+----+-----+--------+-------+------+------------+-----------------------------+-------------+

[2026-06-09T10:32:07] Conteo de nulos por columna: {'data': 0, 'grupobiologico': 0, 'fao': 0, 'especie': 0, 'provincia': 0, 'zona': 0, 'lonxa': 0, 'cantidad': 0, 'importe': 0, 'precio': 0, 'anio_archivo': 0, '_grupobiologico_in

### Reglas de calidad

En este bloque se definen las reglas del data contract y se clasifican
los registros según su calidad.

- **REJECT**: registros con errores en datos obligatorios, valores fuera
  de rango o incumplimiento de catálogos de referencia.

- **QUARANTINE**: registros válidos cuyos datos están fuera del rango
  temporal de interés del proyecto (2015-01-01 a 2025-12-31).

- **VALIDO**: registros que cumplen todas las reglas de calidad y pasan
  a la capa silver.

In [46]:
# El orden importa: la primera condición que se cumple decide el destino.
reglas_calidad = [
    # --- REJECT: nulos en campos obligatorios ---
    ("nulo_data",             col("data").isNull(),                                "REJECT"),
    ("nulo_grupobiologico",   col("grupobiologico").isNull(),                      "REJECT"),
    ("fao_nulo_o_vacio",      col("fao").isNull() | (col("fao") == ""),            "REJECT"),
    ("nulo_especie",          col("especie").isNull(),                             "REJECT"),
    ("nulo_provincia",        col("provincia").isNull(),                           "REJECT"),
    ("nulo_zona",             col("zona").isNull(),                                "REJECT"),
    ("nulo_lonxa",            col("lonxa").isNull(),                               "REJECT"),
    ("nulo_cantidad",         col("cantidad").isNull(),                            "REJECT"),
    ("nulo_importe",          col("importe").isNull(),                             "REJECT"),
    ("nulo_precio",           col("precio").isNull(),                              "REJECT"),
    # --- REJECT: valores fuera de rango de negocio ---
    ("cantidad_invalida",     col("cantidad") <= 0,                                "REJECT"),
    ("precio_invalido",       col("precio") <= 0,                                  "REJECT"),
    ("importe_cero",          col("importe") == 0,                                 "REJECT"),
    # --- REJECT: catálogos de referencia ---
    ("fao_longitud_invalida", length(col("fao")) != 3,                             "REJECT"),
    ("provincia_invalida",    ~col("provincia").isin(PROVINCIAS_VALIDAS),           "REJECT"),
    ("zona_invalida",         ~col("zona").isin(ZONAS_VALIDAS),                     "REJECT"),
    # --- QUARANTINE: fuera del rango temporal del proyecto ---
    ("fuera_de_rango_temporal",
        (col("data") < to_date(lit(FECHA_MIN))) | (col("data") > to_date(lit(FECHA_MAX))),
        "QUARANTINE"),
]

log("Cobertura de reglas de calidad (conteo independiente; pueden solaparse):")
for nombre, condicion, categoria in reglas_calidad:
    n = df_silver.filter(condicion).count()
    print(f"[{categoria:>10}] {nombre:<25}: {n}")
    log(f"  - [{categoria}] {nombre}: {n} registros")

# ── PASO 2: Enrutado (construido dinámicamente desde la misma lista) ──
cadena = None
for nombre, condicion, categoria in reglas_calidad:
    cadena = (
        when(condicion, categoria) if cadena is None
        else cadena.when(condicion, categoria)
    )
cadena = cadena.otherwise("VALIDO")

df_calidad = df_silver.withColumn("estado_calidad", cadena)

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 53, Finished, Available, Finished, False)

[2026-06-09T10:38:02] Cobertura de reglas de calidad (conteo independiente; pueden solaparse):
[    REJECT] nulo_data                : 0
[2026-06-09T10:38:13]   - [REJECT] nulo_data: 0 registros
[    REJECT] nulo_grupobiologico      : 0
[2026-06-09T10:38:19]   - [REJECT] nulo_grupobiologico: 0 registros
[    REJECT] fao_nulo_o_vacio         : 0
[2026-06-09T10:38:27]   - [REJECT] fao_nulo_o_vacio: 0 registros
[    REJECT] nulo_especie             : 0
[2026-06-09T10:38:42]   - [REJECT] nulo_especie: 0 registros
[    REJECT] nulo_provincia           : 0
[2026-06-09T10:38:48]   - [REJECT] nulo_provincia: 0 registros
[    REJECT] nulo_zona                : 0
[2026-06-09T10:38:57]   - [REJECT] nulo_zona: 0 registros
[    REJECT] nulo_lonxa               : 0
[2026-06-09T10:39:10]   - [REJECT] nulo_lonxa: 0 registros
[    REJECT] nulo_cantidad            : 0
[2026-06-09T10:39:22]   - [REJECT] nulo_cantidad: 0 registros
[    REJECT] nulo_importe             : 0
[2026-06-09T10:39:39]   - [REJECT

In [47]:
df_silver_ok  = df_calidad.filter(col("estado_calidad") == "VALIDO")
df_quarantine = df_calidad.filter(col("estado_calidad") == "QUARANTINE")
df_reject     = df_calidad.filter(col("estado_calidad") == "REJECT")

# Eliminar columna auxiliar de estado
df_silver_ok  = df_silver_ok.drop("estado_calidad")
df_quarantine = df_quarantine.drop("estado_calidad")
df_reject     = df_reject.drop("estado_calidad")

print(f"Total      : {df_calidad.count()}")
print(f"Silver     : {df_silver_ok.count()}")
print(f"Quarantine : {df_quarantine.count()}")
print(f"Reject     : {df_reject.count()}")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 54, Finished, Available, Finished, False)

Total      : 2131500
Silver     : 2131116
Quarantine : 0
Reject     : 384


In [49]:
# ==============================================================================
# 4. CARGA (Escritura en Capa Silver - Delta Parquet)
# ==============================================================================
# Activamos V-Order: Algoritmo de compresión de Fabric para latencia cero en Power BI
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
log("Carga: escribiendo tabla Delta 'ventas_silver' (mode=overwrite, partitionBy=anio_archivo, V-Order=on)")
# Guardamos como tabla particionada si el volumen lo requiere (ej. por año)
df_silver_ok.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("anio_archivo") \
    .saveAsTable("ventas_silver")

df_quarantine \
    .withColumn("timestamp_revision", current_timestamp()) \
    .write.format("delta").mode("append").saveAsTable("ventas_silver_quarantine")
    
df_reject \
    .withColumn("timestamp_rechazo", current_timestamp()) \
    .write.format("delta").mode("append").saveAsTable("ventas_silver_rejected")

print("Se guarda estos Datos validados, tipados y almacenados en Delta Lake con el nombre de ventas_silver ")
log("FIN: tabla 'ventas_silver' guardada correctamente")

StatementMeta(, 22c7dc9f-0076-43c6-a71c-b7e6cfbd4f3e, 56, Finished, Available, Finished, False)

[2026-06-09T10:46:37] Carga: escribiendo tabla Delta 'ventas_silver' (mode=overwrite, partitionBy=anio_archivo, V-Order=on)
Se guarda estos Datos validados, tipados y almacenados en Delta Lake con el nombre de ventas_silver 
[2026-06-09T10:48:24] FIN: tabla 'ventas_silver' guardada correctamente


In [12]:
from notebookutils import mssparkutils

# Esto sí apaga el clúster físico desde el código
#mssparkutils.session.stop()

StatementMeta(, 1a707a11-ad64-466d-95f6-7a503491acb8, 14, Finished, Available, Finished, False)